# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/titlyzaman25/flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import pandas as pd
import numpy as np

HF_TOKEN = userdata.get("HF_TOKEN")

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

df = pd.read_parquet(march_file)

print("March rows:", len(df))
print("Columns:")
print(df.columns.tolist())

March rows: 9841378
Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [2]:
# Download supporting warehouse tables

content_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

query_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_query_90d.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

content_df = pd.read_parquet(content_file)
query_df = pd.read_parquet(query_file)

print("Content rows:", len(content_df))
print("Query rows:", len(query_df))

Content rows: 519606
Query rows: 2414248


In [3]:
# Merge March performance with content metadata

baseline = df.merge(
    content_df,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
    suffixes=("", "_content")
)

print("March performance rows:", len(df))
print("After metadata merge:", len(baseline))
print(
    "Unique content items:",
    baseline["content_hash_id"].nunique()
)

March performance rows: 9841378
After metadata merge: 9841378
Unique content items: 331437


In [4]:
# Verify the March performance grain

daily_grain = (
    baseline
    .groupby(
        ["report_date", "client_hash_id", "content_hash_id"]
    )
    .size()
)

print("Total rows:", len(baseline))
print("Unique date-client-content combinations:", len(daily_grain))
print("Maximum rows per date-client-content:", daily_grain.max())
print(
    "Duplicate date-client-content combinations:",
    (daily_grain > 1).sum()
)

Total rows: 9841378
Unique date-client-content combinations: 9841378
Maximum rows per date-client-content: 1
Duplicate date-client-content combinations: 0


In [5]:
# Aggregate March performance to one row per content item

content_perf = (
    baseline
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        gsc_impressions=("gsc_impressions", "sum"),
        gsc_clicks=("gsc_clicks", "sum"),
        gsc_avg_position=("gsc_avg_position", "mean")
    )
)

print("Content-level rows:", len(content_perf))
print(
    "Unique content items:",
    content_perf["content_hash_id"].nunique()
)

Content-level rows: 331437
Unique content items: 331437


In [6]:
# Attach content metadata to the content-level performance table

metadata_cols = [
    "client_hash_id",
    "content_hash_id",
    "content_created_date",
    "content_updated_date",
    "last_optimized_date",
    "content_type",
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "main_intent",
    "backlinks",
    "category_count",
    "is_published",
    "is_deleted"
]

content_meta = content_df[metadata_cols].copy()

baseline = content_perf.merge(
    content_meta,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

print("Final baseline rows:", len(baseline))
print(
    "Unique content items:",
    baseline["content_hash_id"].nunique()
)

Final baseline rows: 331437
Unique content items: 331437


In [7]:
# Prepare dates for the March 2026 decision point

decision_date = pd.Timestamp("2026-03-31")

baseline["content_created_date"] = pd.to_datetime(
    baseline["content_created_date"],
    errors="coerce"
)

baseline["content_updated_date"] = pd.to_datetime(
    baseline["content_updated_date"],
    errors="coerce"
)

baseline["last_optimized_date"] = pd.to_datetime(
    baseline["last_optimized_date"],
    errors="coerce"
)

print("Decision date:", decision_date)

Decision date: 2026-03-31 00:00:00


In [8]:
# Check for metadata dates after the March decision date

for col in [
    "content_created_date",
    "content_updated_date",
    "last_optimized_date"
]:
    future_count = (
        baseline[col].notna()
        & (baseline[col] > decision_date)
    ).sum()

    print(f"{col} | future values: {future_count}")

content_created_date | future values: 2124
content_updated_date | future values: 293358
last_optimized_date | future values: 42517


In [9]:
# SIGNAL 1 — STALENESS
# Only use content_updated_date if it was known by March 31, 2026.

valid_update = (
    baseline["content_updated_date"].notna()
    & (baseline["content_updated_date"] <= decision_date)
)

baseline["days_since_update"] = np.where(
    valid_update,
    (
        decision_date
        - baseline["content_updated_date"]
    ).dt.days,
    np.nan
)

baseline["staleness_bucket"] = pd.cut(
    baseline["days_since_update"],
    bins=[-1, 90, 180, np.inf],
    labels=[
        "0-90 days",
        "91-180 days",
        "181+ days"
    ]
)

print("Staleness buckets:")
print(
    baseline["staleness_bucket"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nValid update dates:", valid_update.sum())
print(
    "Unknown/future update dates:",
    (~valid_update).sum()
)

Staleness buckets:
staleness_bucket
0-90 days       30655
91-180 days      3608
181+ days        3816
NaN            293358
Name: count, dtype: int64

Valid update dates: 38079
Unknown/future update dates: 293358


In [10]:
# SIGNAL 2 — CTR VS POSITION

# Calculate March content-level CTR
baseline["ctr"] = np.where(
    baseline["gsc_impressions"] > 0,
    baseline["gsc_clicks"] / baseline["gsc_impressions"],
    np.nan
)

# Create position buckets
baseline["position_group"] = pd.cut(
    baseline["gsc_avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=[
        "1-3",
        "4-10",
        "11-20",
        "21+"
    ],
    include_lowest=True
)

print("Position buckets:")
print(
    baseline["position_group"]
    .value_counts(dropna=False)
)

print("\nMean CTR by position:")
print(
    baseline
    .groupby("position_group", observed=True)["ctr"]
    .mean()
)

Position buckets:
position_group
NaN      154699
4-10      81988
21+       44969
11-20     32203
1-3       17578
Name: count, dtype: int64

Mean CTR by position:
position_group
1-3      0.012399
4-10     0.004926
11-20    0.003211
21+      0.001928
Name: ctr, dtype: float64


In [13]:
# Calculate expected CTR by position group

group_stats = (
    baseline
    .groupby("position_group", observed=True)["ctr"]
    .agg(["sum", "count"])
)

# Convert mapped values to numeric
baseline["group_ctr_sum"] = pd.to_numeric(
    baseline["position_group"]
    .map(group_stats["sum"]),
    errors="coerce"
)

baseline["group_ctr_count"] = pd.to_numeric(
    baseline["position_group"]
    .map(group_stats["count"]),
    errors="coerce"
)

# Expected CTR for each row
baseline["expected_ctr"] = (
    baseline["group_ctr_sum"] - baseline["ctr"]
) / (
    baseline["group_ctr_count"] - 1
)

# Flag rows below their position-group expected CTR
baseline["low_ctr"] = (
    baseline["ctr"].notna()
    & baseline["expected_ctr"].notna()
    & (baseline["ctr"] < baseline["expected_ctr"])
)

print("Rows with valid CTR:",
      baseline["ctr"].notna().sum())

print("Rows with valid position group:",
      baseline["position_group"].notna().sum())

print("Rows with expected CTR:",
      baseline["expected_ctr"].notna().sum())

print("Rows below expected CTR:",
      baseline["low_ctr"].sum())

Rows with valid CTR: 176738
Rows with valid position group: 176738
Rows with expected CTR: 176738
Rows below expected CTR: 150830


In [14]:
# Check the CTR comparison

print("\nPosition groups:")
print(baseline["position_group"].value_counts(dropna=False))

print("\nExample CTR comparison:")

print(
    baseline[
        [
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "position_group",
            "ctr",
            "expected_ctr",
            "low_ctr"
        ]
    ].head(20)
)


Position groups:
position_group
NaN      154699
4-10      81988
21+       44969
11-20     32203
1-3       17578
Name: count, dtype: int64

Example CTR comparison:
    gsc_impressions  gsc_clicks  gsc_avg_position position_group       ctr  \
0                 0           0               NaN            NaN       NaN   
1                 0           0               NaN            NaN       NaN   
2                 0           0               NaN            NaN       NaN   
3                 1           0          9.000000           4-10  0.000000   
4                 0           0               NaN            NaN       NaN   
5                 0           0               NaN            NaN       NaN   
6                 0           0               NaN            NaN       NaN   
7               331           2         14.129210          11-20  0.006042   
8                33           0          9.225529           4-10  0.000000   
9                 0           0               NaN       

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
Rule: Prioritize content for human review when it is stale and/or has unusually low CTR for its search position. The score gives one point for staleness and one point for low CTR, so content with both issues is ranked first.

Reason codes: stale_low_ctr, stale, low_ctr, and no_issue.

stale means the last known update was at least 181 days before the March 31, 2026 decision date. low_ctr means the observed CTR is below the expected CTR for its position bucket.

In [15]:
# Section 1 — Encode the baseline rule signals

baseline["stale"] = (
    baseline["days_since_update"] >= 181
).astype(int)

baseline["low_ctr_flag"] = (
    baseline["low_ctr"]
).astype(int)

# Transparent score: 1 point for each signal
baseline["score"] = (
    baseline["stale"] +
    baseline["low_ctr_flag"]
)

# Assign exactly one reason code
baseline["reason_code"] = np.select(
    [
        (baseline["stale"] == 1) & (baseline["low_ctr_flag"] == 1),
        (baseline["stale"] == 1),
        (baseline["low_ctr_flag"] == 1)
    ],
    [
        "stale_low_ctr",
        "stale",
        "low_ctr"
    ],
    default="no_issue"
)

print("Score distribution:")
print(baseline["score"].value_counts().sort_index())

print("\nReason codes:")
print(baseline["reason_code"].value_counts())

Score distribution:
score
0    177046
1    154136
2       255
Name: count, dtype: int64

Reason codes:
reason_code
no_issue         177046
low_ctr          150575
stale              3561
stale_low_ctr       255
Name: count, dtype: int64


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2 — Build the ranked action queue

# Action label: review anything with at least one warning signal
baseline["action"] = np.where(
    baseline["score"] > 0,
    "review",
    "no_action"
)

# Rank highest score first.
# Within the same score, prioritize higher impressions.
baseline = baseline.sort_values(
    by=["score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

baseline["rank"] = np.arange(1, len(baseline) + 1)

# Select the fields needed in the ranked queue
output_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "score",
    "reason_code",
    "action",
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "gsc_avg_position",
    "days_since_update"
]

queue = baseline[output_columns].copy()

# Write the required CSV
output_path = "work/outputs/baseline_action_score.csv"

queue.to_csv(
    output_path,
    index=False
)

print("CSV written:", output_path)
print("Rows written:", len(queue))

print("\nAction labels:")
print(queue["action"].value_counts())

print("\nTop 20:")
print(queue.head(20).to_string(index=False))


CSV written: work/outputs/baseline_action_score.csv
Rows written: 331437

Action labels:
action
no_action    177046
review       154391
Name: count, dtype: int64

Top 20:
 rank          client_hash_id          content_hash_id  score   reason_code action  gsc_impressions  gsc_clicks      ctr  gsc_avg_position  days_since_update
    1 client_c182d11e4862a37d content_42ce26be1ec6be00      2 stale_low_ctr review             4411           6 0.001360          4.262553              264.0
    2 client_c182d11e4862a37d content_bea86ce3455100b0      2 stale_low_ctr review             3670           1 0.000272          6.555793              232.0
    3 client_c182d11e4862a37d content_5120dcbbb086843d      2 stale_low_ctr review             1429           0 0.000000          6.321173              247.0
    4 client_65de48885f4ef01b content_eba53d72e18a9f93      2 stale_low_ctr review              734           2 0.002725          5.234187              231.0
    5 client_65de48885f4ef01b content_c

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
### Top-20 review

1. **Rank 1:** Action — review. Reason — `stale_low_ctr`. Confidence — High because the content has 4,411 impressions, position 4.26, CTR 0.00136, and has not been updated for 264 days. Wrong if the low CTR is acceptable for this specific query or the content is intentionally unchanged.

2. **Rank 2:** Action — review. Reason — `stale_low_ctr`. Confidence — High because it has 3,670 impressions, position 6.56, CTR 0.00027, and is 232 days since update. Wrong if the low CTR is expected for the content's search intent.

3. **Rank 3:** Action — review. Reason — `stale_low_ctr`. Confidence — High because it has 1,429 impressions, position 6.32, zero clicks, and is 247 days old. Wrong if the impressions are too concentrated or the page is not intended to attract clicks.

4. **Rank 4:** Action — review. Reason — `stale_low_ctr`. Confidence — High because it has 734 impressions, position 5.23, CTR 0.00273, and is 231 days old. Wrong if the observed CTR is normal for its query mix.

5. **Rank 5:** Action — review. Reason — `stale_low_ctr`. Confidence — Medium because it is 231 days old with zero clicks, but its position is 26.36 and it has only 592 impressions. Wrong if poor visibility, rather than content quality, explains the low CTR.

6. **Rank 6:** Action — review. Reason — `stale_low_ctr`. Confidence — Medium because it has 486 impressions, position 6.90, CTR 0.00412, and is 234 days old. Wrong if this CTR is not meaningfully below the position-group expectation.

7. **Rank 7:** Action — review. Reason — `stale_low_ctr`. Confidence — Medium because it has 482 impressions, position 7.32, CTR 0.00415, and is 234 days old. Wrong if the lower CTR is caused by query or SERP characteristics rather than stale content.

8. **Rank 8:** Action — review. Reason — `stale_low_ctr`. Confidence — High because it has 410 impressions, zero clicks, position 15.65, and is 231 days old. Wrong if the low CTR is mainly caused by its relatively low search position.

9. **Rank 9:** Action — review. Reason — `stale_low_ctr`. Confidence — Medium because it has 337 impressions, position 8.17, CTR 0.00297, and is 234 days old. Wrong if the CTR difference is caused by query intent rather than content staleness.

10. **Rank 10:** Action — review. Reason — `stale_low_ctr`. Confidence — High because it has 231 impressions, zero clicks, position 6.23, and is 235 days old. Wrong if zero clicks resulted from a small or unusual sample.

11. **Rank 11:** Action — review. Reason — `stale_low_ctr`. Confidence — High because it has 218 impressions, zero clicks, position 7.97, and is 235 days old. Wrong if the low click count is due to limited observations.

12. **Rank 12:** Action — review. Reason — `stale_low_ctr`. Confidence — Medium because it has 212 impressions, zero clicks, position 11.89, and is 234 days old. Wrong if its lower position explains the zero-click result.

13. **Rank 13:** Action — review. Reason — `stale_low_ctr`. Confidence — High because it has 205 impressions, zero clicks, position 4.13, and is 246 days old. Wrong if the search result itself is unattractive despite good ranking.

14. **Rank 14:** Action — review. Reason — `stale_low_ctr`. Confidence — Medium because it has 130 impressions, zero clicks, position 7.18, and is 235 days old. Wrong if the sample is too small to make the CTR signal reliable.

15. **Rank 15:** Action — review. Reason — `stale_low_ctr`. Confidence — Low because it has only 96 impressions, zero clicks, and position 36.63 despite being 243 days old. Wrong if poor ranking, rather than stale content, is the main problem.

16. **Rank 16:** Action — review. Reason — `stale_low_ctr`. Confidence — Medium because it has 86 impressions, zero clicks, position 6.67, and is 232 days old. Wrong if the small impression count makes the CTR unstable.

17. **Rank 17:** Action — review. Reason — `stale_low_ctr`. Confidence — Low because it has only 85 impressions, zero clicks, position 76.50, and is 243 days old. Wrong if the very low position is the main reason for no clicks.

18. **Rank 18:** Action — review. Reason — `stale_low_ctr`. Confidence — Low because it has only 75 impressions, zero clicks, position 74.32, and is 243 days old. Wrong if low visibility, rather than stale content, explains the result.

19. **Rank 19:** Action — review. Reason — `stale_low_ctr`. Confidence — Medium because it has 69 impressions, zero clicks, position 4.87, and is 261 days old. Wrong if the small number of impressions is insufficient to establish a reliable CTR problem.

20. **Rank 20:** Action — review. Reason — `stale_low_ctr`. Confidence — Medium because it has 65 impressions, zero clicks, position 8.03, and is 204 days old. Wrong if the sample is too small or the query intent naturally produces few clicks.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 — Display the top 20 for manual review

top20 = queue.head(20).copy()

print("TOP 20 BASELINE REVIEW")
print("=" * 80)

for _, row in top20.iterrows():
    print(
        f"Rank {int(row['rank'])} | "
        f"Action: {row['action']} | "
        f"Reason: {row['reason_code']} | "
        f"Score: {row['score']} | "
        f"Impressions: {row['gsc_impressions']} | "
        f"CTR: {row['ctr']:.6f} | "
        f"Position: {row['gsc_avg_position']} | "
        f"Days since update: {row['days_since_update']}"
    )

TOP 20 BASELINE REVIEW
Rank 1 | Action: review | Reason: stale_low_ctr | Score: 2 | Impressions: 4411 | CTR: 0.001360 | Position: 4.262552671572721 | Days since update: 264.0
Rank 2 | Action: review | Reason: stale_low_ctr | Score: 2 | Impressions: 3670 | CTR: 0.000272 | Position: 6.555793229775013 | Days since update: 232.0
Rank 3 | Action: review | Reason: stale_low_ctr | Score: 2 | Impressions: 1429 | CTR: 0.000000 | Position: 6.3211729843775935 | Days since update: 247.0
Rank 4 | Action: review | Reason: stale_low_ctr | Score: 2 | Impressions: 734 | CTR: 0.002725 | Position: 5.234186822547582 | Days since update: 231.0
Rank 5 | Action: review | Reason: stale_low_ctr | Score: 2 | Impressions: 592 | CTR: 0.000000 | Position: 26.359738615037447 | Days since update: 231.0
Rank 6 | Action: review | Reason: stale_low_ctr | Score: 2 | Impressions: 486 | CTR: 0.004115 | Position: 6.897523307977751 | Days since update: 234.0
Rank 7 | Action: review | Reason: stale_low_ctr | Score: 2 | Impre

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
### Weak picks + leakage check

Some weak picks are content with very few impressions or very poor search positions. In these cases, low CTR may reflect limited measurement or low visibility rather than a genuine content problem.

The baseline uses only March 2026 performance signals and update information known by the March 31, 2026 decision point. Future `content_updated_date` values are excluded from the staleness calculation. No label-derived fields, future outcomes, or product decision flags are used.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — Weak picks + leakage check

print("=== WEAK PICKS ===")

weak_picks = queue[
    (queue["reason_code"] == "stale_low_ctr") &
    (
        (queue["gsc_impressions"] < 100) |
        (queue["gsc_avg_position"] > 20)
    )
].head(10)

print(weak_picks.to_string(index=False))


print("\n=== LEAKAGE CHECK ===")

used_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "days_since_update"
]

print("Features used by the baseline:")
for feature in used_features:
    print("-", feature)

print("\nFuture-date protection:")

future_update_count = (
    baseline["content_updated_date"].notna()
    & (baseline["content_updated_date"] > decision_date)
).sum()

print(
    "Future content_updated_date values:",
    future_update_count
)

print(
    "Future update dates excluded from staleness:",
    (~valid_update).sum()
)

print("\nLeakage conclusion:")
print(
    "No future-window or label-derived inputs are used in the baseline score."
)

=== WEAK PICKS ===
 rank          client_hash_id          content_hash_id  score   reason_code action  gsc_impressions  gsc_clicks  ctr  gsc_avg_position  days_since_update
    5 client_65de48885f4ef01b content_c126a43258b574c3      2 stale_low_ctr review              592           0  0.0         26.359739              231.0
   15 client_65de48885f4ef01b content_95d140b7cd7e3897      2 stale_low_ctr review               96           0  0.0         36.629101              243.0
   16 client_65de48885f4ef01b content_2879353f031b1983      2 stale_low_ctr review               86           0  0.0          6.667196              232.0
   17 client_73cda7b4e4f265ea content_3e435be3dc7de7ef      2 stale_low_ctr review               85           0  0.0         76.498430              243.0
   18 client_73cda7b4e4f265ea content_f4098d5b2c2eeb18      2 stale_low_ctr review               75           0  0.0         74.323264              243.0
   19 client_c182d11e4862a37d content_bc5c5010efa1afe0   

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.